In [0]:
CREATE OR REPLACE TABLE silver_customer AS
WITH customer_spending AS (
    SELECT
        c.CustomerId,
        SUM(il.UnitPrice * il.Quantity) AS total_spending
    FROM workspace.default.chinook_customer c
    LEFT JOIN workspace.default.chinook_invoice i ON c.CustomerId = i.CustomerId
    LEFT JOIN workspace.default.chinook_invoiceline il ON i.InvoiceId = il.InvoiceId
    GROUP BY c.CustomerId
),
percentile_thresholds AS (
    SELECT
        PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spending) AS high_threshold,
        PERCENTILE_CONT(0.60) WITHIN GROUP (ORDER BY total_spending) AS medium_threshold
    FROM customer_spending
)
SELECT
    c.CustomerId,
    TRIM(c.FirstName) AS FirstName,
    TRIM(c.LastName) AS LastName,
    TRIM(c.Company) AS Company,
    TRIM(c.Country) AS Country,
    TRIM(c.Email) AS Email,
    c.SupportRepId,
    CASE
        WHEN COALESCE(cs.total_spending, 0) >= pt.high_threshold THEN 'High'
        WHEN COALESCE(cs.total_spending, 0) >= pt.medium_threshold THEN 'Medium'
        ELSE 'Low'
    END AS SpendingTier
FROM workspace.default.chinook_customer c
LEFT JOIN customer_spending cs ON c.CustomerId = cs.CustomerId
CROSS JOIN percentile_thresholds pt
WHERE c.CustomerId IS NOT NULL
  AND c.Country IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.CustomerId ORDER BY c.LastName, c.FirstName) = 1;
